# T08 — Training UNet Attention su LOL-v2 Real

Addestra `UNetAttention` (base_channels=32, ~7.77M parametri) sulla partizione
**Real_captured** di LOL-v2. **Stesso identico protocollo della baseline (T07)**
per un confronto equo: stessi split, stessi iperparametri, stesso seed.

| | |
|---|---|
| Dataset | LOL-v2 Real_captured |
| Train | 90% dei 689 sample ufficiali (≈620 immagini) |
| Val | 10% dei 689 sample ufficiali (≈69 immagini) |
| Test | 100 sample ufficiali (cartella Test — mai visti durante il training) |
| Hardware target | Kaggle T4 (16 GB VRAM) |
| Checkpoint output | `checkpoints/attention/best.pt` e `last.pt` |

**Differenze rispetto alla baseline:**
- Modello: `UNetAttention` (CBAM dopo ogni blocco decoder)
- `experiment_name = "attention"`
- Tutto il resto è identico → confronto diretto possibile

In [ ]:
# ── Installazione dipendenze mancanti (necessario su Kaggle) ──────────────
import subprocess, sys

def _pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", *pkgs, "-q"])

try:
    import piq
except ImportError:
    _pip("piq>=0.8.0")

try:
    import torchinfo
except ImportError:
    _pip("torchinfo>=1.8.0")

print("Dipendenze OK")

In [ ]:
# ── Import ────────────────────────────────────────────────────────────────
import os
import sys
from pathlib import Path

import torch
from torch.utils.data import DataLoader, Subset

# ── Rilevamento ambiente ──────────────────────────────────────────────────
ON_KAGGLE = Path("/kaggle/working").exists()

print(f"ON_KAGGLE : {ON_KAGGLE}")
print(f"CUDA      : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU       : {torch.cuda.get_device_name(0)}")
    print(f"VRAM      : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ── Costanti ──────────────────────────────────────────────────────────────
if ON_KAGGLE:
    PROJECT_ROOT = Path("/kaggle/input/datasets/mattiacagnetta/low-light-src")
    OUTPUT_ROOT  = Path("/kaggle/working")
    LOLV2_ROOT   = Path("/kaggle/input/datasets/mattiacagnetta/lol-v2/LOL-v2")
else:
    PROJECT_ROOT = Path("..").resolve()
    OUTPUT_ROOT  = PROJECT_ROOT
    LOLV2_ROOT   = PROJECT_ROOT / "data" / "LOL-v2"

# Aggiorna sys.path con il PROJECT_ROOT corretto
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f"sys.path[0] = {sys.path[0]}")

# Percorsi cartelle LOL-v2 Real_captured
TRAIN_LOW    = LOLV2_ROOT / "Real_captured" / "Train" / "Low"
TRAIN_NORMAL = LOLV2_ROOT / "Real_captured" / "Train" / "Normal"
TEST_LOW     = LOLV2_ROOT / "Real_captured" / "Test"  / "Low"
TEST_NORMAL  = LOLV2_ROOT / "Real_captured" / "Test"  / "Normal"

# Cartelle output
SPLITS_DIR = OUTPUT_ROOT / "data" / "splits"
CKPT_DIR   = OUTPUT_ROOT / "checkpoints"
LOG_DIR    = OUTPUT_ROOT / "runs"

# Iperparametri training (identici alla baseline per confronto equo)
BATCH_SIZE  = 8
NUM_WORKERS = 2
IMG_SIZE    = 256

# Verifica percorsi
for p in [TRAIN_LOW, TRAIN_NORMAL, TEST_LOW, TEST_NORMAL]:
    status = "\u2713" if p.exists() else "\u2717 NON TROVATO"
    print(f"  {status}  {p}")

In [ ]:
# ── Split deterministici train / val / test ───────────────────────────────
# Identici alla baseline: stesso seed=42, stessa val_fraction=0.10.
# Se i file di split esistono gia' (da T07), vengono sovrascritti con valori
# identici — non cambia nulla.

from src.data.dataset import PairedImageDataset
from src.data.splits  import create_and_save_train_val_split, save_test_split

lolv2_key = lambda s: s.lstrip("abcdefghijklmnopqrstuvwxyz")

full_train_ds = PairedImageDataset(TRAIN_LOW, TRAIN_NORMAL, key_fn=lolv2_key)
test_ds_raw   = PairedImageDataset(TEST_LOW,  TEST_NORMAL,  key_fn=lolv2_key)

print(f"Train ufficiale: {len(full_train_ds)} coppie")
print(f"Test ufficiale : {len(test_ds_raw)} coppie")

SPLITS_DIR.mkdir(parents=True, exist_ok=True)
splits = create_and_save_train_val_split(
    full_train_ds.stems,
    splits_dir=SPLITS_DIR,
    split_name="lolv2_real",
    val_fraction=0.10,
    seed=42,
)
save_test_split(test_ds_raw.stems, SPLITS_DIR / "lolv2_real_test.txt")

train_stems = splits["train"]
val_stems   = splits["val"]
print(f"\nSplit salvati in: {SPLITS_DIR}")

In [ ]:
# ── Dataset e DataLoader ──────────────────────────────────────────────────
from src.data.transforms import get_preprocessing_transform, get_paired_augmentation

preprocess = get_preprocessing_transform(IMG_SIZE)
augment    = get_paired_augmentation(IMG_SIZE)

train_ds_full = PairedImageDataset(
    TRAIN_LOW, TRAIN_NORMAL,
    key_fn=lolv2_key,
    paired_transform=augment,
    transform=preprocess,
)
eval_ds_full = PairedImageDataset(
    TRAIN_LOW, TRAIN_NORMAL,
    key_fn=lolv2_key,
    transform=preprocess,
)

train_set = set(train_stems)
val_set   = set(val_stems)
train_indices = [i for i, s in enumerate(train_ds_full.stems) if s in train_set]
val_indices   = [i for i, s in enumerate(eval_ds_full.stems)  if s in val_set]

train_subset = Subset(train_ds_full, train_indices)
val_subset   = Subset(eval_ds_full,  val_indices)

train_loader = DataLoader(
    train_subset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=(NUM_WORKERS > 0),
    drop_last=True,
)
val_loader = DataLoader(
    val_subset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=(NUM_WORKERS > 0),
)

print(f"Train samples : {len(train_subset)}  ({len(train_loader)} batch x {BATCH_SIZE})")
print(f"Val samples   : {len(val_subset)}  ({len(val_loader)} batch x {BATCH_SIZE})")

batch = next(iter(train_loader))
print(f"\nBatch shape : low={tuple(batch['low'].shape)}  normal={tuple(batch['normal'].shape)}")
print(f"Range       : [{batch['low'].min():.3f}, {batch['low'].max():.3f}]")

In [ ]:
# ── Modello, Loss e TrainConfig ───────────────────────────────────────────
# UNICA DIFFERENZA rispetto a T07: UNetAttention invece di UNetBaseline
# e experiment_name="attention".
# Tutti gli altri iperparametri sono identici per garantire un confronto equo.

from src.models  import UNetAttention          # <-- variante con CBAM
from src.losses  import CombinedLoss
from src.train   import TrainConfig, Trainer

model     = UNetAttention(base_channels=32)    # <-- variante con CBAM
criterion = CombinedLoss(alpha=0.8)

config = TrainConfig(
    # ── Ottimizzatore ────────────────────────────────────────────────────
    optimizer_name          = "adamw",
    lr                      = 1e-4,
    weight_decay            = 1e-4,
    grad_clip_norm          = 1.0,

    # ── Schedule ─────────────────────────────────────────────────────────
    epochs                  = 100,
    scheduler_name          = "cosine",
    early_stopping_patience = 15,

    # ── Validazione e logging ─────────────────────────────────────────────
    val_every_n_epochs      = 1,
    log_every_n_epochs      = 1,
    n_visual_samples        = 4,

    # ── Hardware e riproducibilita' ───────────────────────────────────────
    amp                     = True,
    device                  = "auto",
    seed                    = 42,

    # ── Checkpoint e log ─────────────────────────────────────────────────
    checkpoint_dir          = str(CKPT_DIR),
    log_dir                 = str(LOG_DIR),
    experiment_name         = "attention",     # <-- cartella separata
)

print(model)
print(f"\nParametri    : {sum(p.numel() for p in model.parameters()):,}")
print(f"\nConfig:")
for field, value in vars(config).items():
    print(f"  {field:<28} = {value}")

In [ ]:
# ── Training ──────────────────────────────────────────────────────────────
trainer = Trainer(model, train_loader, val_loader, criterion, config)
trainer.fit()

In [ ]:
# ── Riepilogo finale ──────────────────────────────────────────────────────
import json
from dataclasses import asdict

best_ckpt = trainer.ckpt_dir / "best.pt"
last_ckpt = trainer.ckpt_dir / "last.pt"

print("=" * 60)
print("RIEPILOGO TRAINING — UNetAttention")
print("=" * 60)
print(f"Epoche completate : {trainer.epoch}")
print(f"Best val_loss     : {trainer.best_score:.6f}")
print(f"Checkpoint best   : {best_ckpt}  ({best_ckpt.stat().st_size / 1e6:.1f} MB)")
print()

# Valutazione finale con il best checkpoint
ckpt = torch.load(best_ckpt, map_location=trainer.device, weights_only=False)
trainer.model.load_state_dict(ckpt["model"])
final_metrics = trainer.val_epoch()

print("Metriche val (best checkpoint):")
print(f"  val_loss : {final_metrics['loss']:.6f}")
print(f"  PSNR     : {final_metrics['psnr']:.2f} dB")
print(f"  SSIM     : {final_metrics['ssim']:.4f}")
print()

# Confronto diretto con la baseline (se disponibile)
baseline_summary = CKPT_DIR / "baseline" / "training_summary.json"
if baseline_summary.exists():
    bs = json.loads(baseline_summary.read_text(encoding="utf-8"))
    print("Confronto Baseline vs Attention (val):")
    print(f"  {'':20} {'Baseline':>12} {'Attention':>12} {'Delta':>10}")
    print(f"  {'PSNR (dB)':20} {bs['val_psnr_db']:>12.2f} {final_metrics['psnr']:>12.2f} {final_metrics['psnr']-bs['val_psnr_db']:>+10.2f}")
    print(f"  {'SSIM':20} {bs['val_ssim']:>12.4f} {final_metrics['ssim']:>12.4f} {final_metrics['ssim']-bs['val_ssim']:>+10.4f}")
    print(f"  {'val_loss':20} {bs['best_val_loss']:>12.6f} {final_metrics['loss']:>12.6f} {final_metrics['loss']-bs['best_val_loss']:>+10.6f}")
    print()

# Salva summary
summary = {
    "experiment"    : config.experiment_name,
    "model"         : str(trainer.model),
    "epochs_done"   : trainer.epoch,
    "best_val_loss" : trainer.best_score,
    "val_psnr_db"   : final_metrics["psnr"],
    "val_ssim"      : final_metrics["ssim"],
    "hardware"      : torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
    "config"        : asdict(config),
}
(trainer.ckpt_dir / "training_summary.json").write_text(
    json.dumps(summary, indent=2), encoding="utf-8"
)
print(f"Summary salvato in: {trainer.ckpt_dir / 'training_summary.json'}")

# Zip per download su Kaggle
if ON_KAGGLE:
    import shutil
    from IPython.display import FileLink, display
    zip_path = "/kaggle/working/attention_checkpoint.zip"
    shutil.make_archive(
        "/kaggle/working/attention_checkpoint", "zip",
        root_dir=str(trainer.ckpt_dir.parent),
        base_dir="attention",
    )
    print(f"\nZip pronto per il download:")
    display(FileLink("attention_checkpoint.zip"))